## Lab 02 / Step 2 : Supervised Fine-Tuning (SFT) of Large Language Models (LLMs)

## Task : from a specific prompt, generate a response

## Use pre-trained SSL-LLM model from Step 1 and fine-tuned it with a training set of (prompt,response)

### Xavier Bresson

### Number of data points for GPT-3, 175B parameters
+ Step #1 : 300B tokens
+ Step #2 : 10K-100k pairs (prompt, response)
+ Step #3 : 100K-1M triples (prompt, positive response, negative response)
+ Step #4 : 10K-100K prompts

### Number of data points for this tutorial
+ Step #1 : 200M tokens
+ Step #2 : 200K pairs (prompt, response)
+ Step #3 : 100K triples (prompt, positive response, negative response)
+ Step #4 : 100 prompts

### Objectives
+ Supervised learning with auto-regressive prediction of sequences
+ Train with batch of pairs (prompt, response) for fast training with GPU
+ Load a pre-trained LLM network


In [1]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/IPAM_Mar26_codes/labs_lecture09'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd
    

In [1]:
# Libraries
import torch
print(torch.__version__)
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime


2.1.2+cu118


## Time stamp for save/load data


In [2]:
# save time stamp
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")

# check dataset folder exists
data_dir = os.path.join('dataset')
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# select a time stamp
use_saved_time_stamp = False
use_saved_time_stamp = True
if use_saved_time_stamp:
    time_stamp = '26-02-07--18-48-32' # trained on GPU on xxx

print('time_stamp:', time_stamp, '\n')


time_stamp: 26-02-23--15-37-33 



## Load dictionary of tokens from Step 1


In [3]:
# load_file_dictionary = 'dataset/step1_SSL_dictionary_' + str(time_stamp) + '.pt'
file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
print('file_dictionary:', file_dictionary, '\n')
import os
os.makedirs('dataset', exist_ok=True)
if not os.path.exists(file_dictionary):
    print(f'Downloading {file_dictionary} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
    !wget -q "{dropbox_url}" -O "{file_dictionary}"
dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary) # load dictionary of tokens
print('dictionary:',dictionary,'\n')
print('num_tokens (unique):',num_tokens,'\n')
print('token2index:', token2index,'\n')
print('index2token:', index2token,'\n')
func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'


file_dictionary: dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14', '17', '32', '59', '10', '12', '16', '80', '35', '33', '45', '0', 'generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', 'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', 'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<PAD>', '<EOS>'] 

num_tokens (unique): 129 

token2index: {'51': 0, '57': 1, '63':

## Generate training tuples/pairs of (prompt, response)

For real-world LLMs like ChatGPT, training pairs are prepared by human experts in English, coding, biology, etc


In [4]:
# generate arithmetic series
max_value = 100 # maximum value in the sequence
def arithmetic_series(max_value, s, d, n):
    seq = []
    for i in range(n):
        v = s + i * d
        if v <= max_value:
            seq.append(v)
        else:
            break
    return seq

# generate training data, i.e. pairs of prompt and response
#   prompt = [ generate arithmetic series of 5 terms with difference 2 starting at 3 ]
# response = [ 3, 5, 7, 9, 11 ]
save_training_data = False
save_training_data = True
if save_training_data:

    # collect high-quality "human" training set
    list_prompt = []
    list_response = []
    num_training_data = 100 # debug
    num_training_data = 200000 # 200K number of pairs of (prompt, response)
    start = time.time()
    for idx in range(num_training_data):

        # parameters for arithmetic series
        m = max_value # maximum value in the sequence
        s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
        d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
        n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
        #print('max_value: %d, start_value: %d, common_difference: %d, number_of_terms: %d' % (max_value,s,d,n))

        # generate prompt : sample a prompt between 3 candidate prompts
        prompt = {}
        prompt[1] = 'generate an arithmetic series with ' + str(n) + ' terms starting with value ' + str(s) \
                    + ' and common difference ' + str(d)
        prompt[2] = 'make a series of arithmetic type which starts at ' + str(s) + ' with ' + str(n) + ' elements and ' + str(d)\
                    + ' common difference value'
        prompt[3] = 'Let ' + str(n) + ' be the number of terms ' + str(s) + ' the starting number and ' + str(d)\
                    + ' the common difference then write the arithmetic series'
        random_int = torch.randint(low=1, high=3+1, size=(1,)).item() # random number in {1,2,3}
        prompt = prompt[random_int]
        response = arithmetic_series(max_value,s,d,n)

        # covert from token to integrer
        prompt = [str(i) for i in prompt.split()] # convert a string into seq of tokens (w/ string type)
        prompt = func_tokens2indices(prompt) # convert from token (str) to index (int)
        prompt = torch.tensor(prompt) # convert to pytorch
        response = [str(i) for i in response] # convert a string into seq of tokens (w/ string type)
        response = func_tokens2indices(response) # convert from token (str) to index (int)
        response = torch.tensor(response) # convert to pytorch

        # append
        list_prompt.append(prompt)
        list_response.append(response)

        # track
        if not idx%1000:
            print('idx: %d, time(sec): %.3f' % (idx, time.time()-start) )

    # print
    print('number of training data (prompt, response) :',len(list_prompt),'\n')
    for idx, (prompt, response) in enumerate(zip(list_prompt[:3],list_response[:3])):
        print('training_set[%d] (pytorch) : %s : %s ' % (idx, prompt, response) )
        prompt = func_tokens2str(func_indices2tokens(prompt.tolist()))
        response = func_tokens2str(func_indices2tokens(response.tolist()))
        print('training_set[%d] (token) : %s : %s ' % (idx, prompt, response) , '\n' )

    # save training data
    save_file = data_dir + '/step2_SFT_indices_training_set_' + time_stamp + '.pt'
    print('save_file:', save_file, '\n')
    torch.save([list_prompt, list_response],save_file) # save list of prompts and sequences

else:

    # load training data
    load_file = data_dir + '/step2_SFT_indices_training_set_' + time_stamp + '.pt'
    print('load_file:', load_file, '\n')
    list_prompt, list_response = torch.load(load_file)

    # print
    print('number of training data (prompt, response) :',len(list_prompt),'\n')
    for idx, (prompt, response) in enumerate(zip(list_prompt[:3],list_response[:3])):
        prompt = func_tokens2str(func_indices2tokens(prompt.tolist()))
        response_token = func_tokens2str(func_indices2tokens(response.tolist()))
        print('training_set[%d] (token) : %s : %s ' % (idx, prompt, response_token) , '\n' )



idx: 0, time(sec): 0.020
idx: 1000, time(sec): 0.073
idx: 2000, time(sec): 0.110
idx: 3000, time(sec): 0.146
idx: 4000, time(sec): 0.181
idx: 5000, time(sec): 0.217
idx: 6000, time(sec): 0.326
idx: 7000, time(sec): 0.363
idx: 8000, time(sec): 0.398
idx: 9000, time(sec): 0.434
idx: 10000, time(sec): 0.470
idx: 11000, time(sec): 0.506
idx: 12000, time(sec): 0.543
idx: 13000, time(sec): 0.579
idx: 14000, time(sec): 0.615
idx: 15000, time(sec): 0.651
idx: 16000, time(sec): 0.687
idx: 17000, time(sec): 0.722
idx: 18000, time(sec): 0.759
idx: 19000, time(sec): 0.795
idx: 20000, time(sec): 0.831
idx: 21000, time(sec): 0.866
idx: 22000, time(sec): 0.902
idx: 23000, time(sec): 0.938
idx: 24000, time(sec): 0.975
idx: 25000, time(sec): 1.011
idx: 26000, time(sec): 1.046
idx: 27000, time(sec): 1.082
idx: 28000, time(sec): 1.118
idx: 29000, time(sec): 1.153
idx: 30000, time(sec): 1.189
idx: 31000, time(sec): 1.226
idx: 32000, time(sec): 1.262
idx: 33000, time(sec): 1.297
idx: 34000, time(sec): 1.33

## Step 2 : Train LLM model by supervised fine-tuning of Step1-LLM

## Dataset is composed of pairs (prompt, response)


In [6]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training 
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation 
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

def get_batch(batch_size, list_prompt_response_idx):
    n_samples = list_prompt_response_idx.size(0)
    perm = torch.randperm(n_samples) # shuffle the current pool
    batch_indices = perm[:batch_size] # First 'batch_size' indices are for the batch
    remaining_indices = perm[batch_size:] # The remaining indices are for the new list
    batch_idx = list_prompt_response_idx[batch_indices]
    new_list_prompt_response_idx = list_prompt_response_idx[remaining_indices]
    return batch_idx, new_list_prompt_response_idx

# Predict next token(t+1) given context = {token(t), token(t-1), ..., token(t-context_length)}
#
#  Prediction is auto-regressive, i.e. one token at a time (different from step #1, one-shot prediction)
#
#  Example of auto-regressive prediction for one SINGLE prompt
#
#           context
#           -------  <= context length (= 4 tokens) to predict next token
#                   |<= starts auto-regressive prediction at end of prompt = 5
#  seq    = 1 2 3 4 5 6 7 8 9 1
#           --------- ----------
#            prompt |  response
#                   |
#                   |<= predicts next_token = 6
#  target =         6 7 8 9 1 eos <= labeled tokens used for loss / training
#                   -------------
#                  len(response)+1 = num_tokens to predict
#                   |
# predicted_seq =   9 4 1 9 2 7   <= predicted tokens (one token at a time) used for loss / training
#
#
#  Example of auto-regressive prediction for a BATCH of prompts
#
# Prepare a batch of sampled (prompt,response)
#  Let the tokens P = <Padding> and E = <EOF> (end-of-file)
#
#                      batch_size (all prompts have the same length with padding if needed) 
#                 ---------------------
# batch_prompt = [ P, P, P, 1, 2, 3, 4 ]  // prompt = [1, 2, 3, 4]
#              = [ 3, 4, 5, 6, 7, 8, 9 ]  // prompt = [3, 4, 5, 6, 7, 8, 9]
#                         ...
#              = [ P, P, P, P, P, 5, 6 ]  // prompt = [5, 6]
#                        -------------
#                           context <= context length (= 5 tokens) to predict next token
#                                    | <= starts auto-regressive prediction at end of batch_seq
#                                    |
# batch_predicted_seq            = [ 9, 0, 5, 3, 5, 1 ]
#                                = [ 2, 5, 1, 7, 9, 3 ]
#                                          ...
#                                = [ 6, 2, 1, 8, 3, 9 ]
#                                   -----------------
#                                       max_len_response+1 (all generated responses have the same length ) 
#                                    |
# batch_target                   = [ 5, 6, 7, 8, E, E ]
#                                = [ 0, 1, 2, E, E, E ]
#                                          ...
#                                = [ 7, 8, 9, 0, 1, E ]

# class of supervised fine-tuning LM network (step 2)
class SFT_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int #torch.tensor([eos_int]).to(device)
        self.eos = eos_int #torch.tensor([eos_int]).to(device)
    def forward(self, batch_idx, list_prompt, list_response): # batch_idx.size=[batch_size], len(list_prompt,list_response), =[num_prompt_response]
        prompts = [list_prompt[idx] for idx in batch_idx] # sample list of prompts, len(prompts)=num_prompt_response
        len_prompt = max([len(prompt) for prompt in prompts]) # compute max of prompt lengths
        responses = [list_response[idx] for idx in batch_idx] # sample list of responses, len(prompts)=num_prompt_response
        len_response = max([len(response) for response in responses]) # compute max of response lengths
        batch_size = batch_idx.size(0)
        predicted_seq = torch.ones(batch_size, max(len_prompt,self.context_length)).long().to(device) * self.padding # initiliaze with padding
        for idx in range(batch_size): predicted_seq[idx, -prompts[idx].size(0):] = prompts[idx] # fill batch_predicted_seq with prompt, right-aligned
        predicted_seq_scores = []
        for idx in range(len_response+1): # number of auto-regressive prediction
            context = predicted_seq[:,-self.context_length:] # size=[batch_size, context_length
            H = self.token2vec(context) + self.PE_embedding(self.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
            for transformer_block in self.transformer_blocks: H = transformer_block(H) # size=(batch_size, context_length, d)
            token_scores = H[:,-1,:] # extract last token to predict the next one, size=[batch_size, d]
            token_scores = self.token_prediction(token_scores) # compute scores, size=[batch_size, num_tokens]
            token_probs = torch.softmax(token_scores, dim=1) # compute probs, size=[batch_size, num_tokens]
            next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[batch_size, 1]
            predicted_seq = torch.cat((predicted_seq, next_token), dim=1) # size=[batch_size, current_seq_len+1]
            predicted_seq_scores.append(token_scores.unsqueeze(1)) # append size=[batch_size, 1, num_tokens]
        predicted_seq_scores = torch.cat(predicted_seq_scores, dim=1) # size=[batch_size, len_response+1, num_tokens]
        score_seq = []
        for idx in range(batch_size): score_seq.append(predicted_seq_scores[idx,:responses[idx].size(0)+1,:]) # append size=[len_response_idx, num_tokens]
        score_seq = torch.cat(score_seq, dim=0) # [num_tokens_for_loss=sum_idx len_response_idx+1, num_tokens]
        target_seq = [] #torch.ones(batch_size, len_response+1).long().to(device) * self.eos # init with padding value
        for idx in range(batch_size):  target_seq.append(responses[idx].to(device)); target_seq.append(self.eos); # append size=[len_response_idx]
        target_seq = torch.cat(target_seq, dim=0) # [num_tokens_for_loss]
        predicted_seq = predicted_seq[:,max(len_prompt,self.context_length):] # size=[batch_size, max(len_prompt,context_length)]
        return score_seq, target_seq, predicted_seq # return prediction sequences and scores w.r.t. the dictionary of tokens

# Load pre-trained SSL-LLM network from Step 1
# checkpoint_file = 'checkpoint/step1_checkpoint_SSL_LLM_' + str(time_stamp) + '.pkl'
import os
checkpoint_file = 'checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl' 
print('checkpoint_file:', checkpoint_file, '\n')
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/dlmsnuk43z9q204ptjtvn/step1_checkpoint_SSL_LLM_26-02-07-18-48-32_200M_10M.pkl?rlkey=owy7mky13kou2i04m60l6jg4c&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}"
checkpoint = torch.load(checkpoint_file, map_location=device)
epoch = checkpoint['epoch']
tot_time = checkpoint['tot_time']
loss = checkpoint['loss']
print('Load pre-trained SSL-LLM: \n checkpoint file: {:s}\n epoch: {:d}, time: {:.3f}min, loss={:.4f}'.format(checkpoint_file,epoch,tot_time,loss))
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
batch_length = net_parameters['batch_length']; context_length = batch_length
num_layers = net_parameters['num_layers']
print(' num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, context_length, num_heads, num_layers) )
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
print('num_tokens: %d, padding_int: %d, eos_int: %d\n' % (num_tokens, padding_int, eos_int))
SFT_LLMnet = SFT_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int)
SFT_LLMnet = SFT_LLMnet.to(device) # GPU training 
SFT_LLMnet.load_state_dict(checkpoint['SSL_LLMnet_dict']) # load pre-trained SSL-LM network from step 1
num_param = number_param(SFT_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# save checkpoint
net_parameters = {}
net_parameters['num_tokens'] = num_tokens
net_parameters['d'] = d
net_parameters['num_heads'] = num_heads
net_parameters['context_length'] = context_length
net_parameters['num_layers'] = num_layers
net_parameters['padding_int'] = padding_int
net_parameters['eos_int'] = eos_int
checkpoint_dir = os.path.join("checkpoint")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
print('checkpoint file :', checkpoint_dir + '/step2_checkpoint_SFT_LLM_' + time_stamp + '.pkl', '\n')

# Optimizer
num_prompt_response = len(list_prompt) # number of prompt+response sequences
batch_size = 3 # debug
batch_size = 100
num_batch = num_prompt_response // batch_size # number of batches
print('num_prompt_response: %d, batch_size: %d, num_batch: %d\n' % (num_prompt_response, batch_size, num_batch))
num_epochs = 1
print('num_epochs: ',num_epochs,'\n')
init_lr = 3e-4
final_lr = 1e-6
warmup_steps = 100
total_steps = num_batch * num_epochs  # Total training steps 
# Optimizer
optimizer = torch.optim.AdamW(SFT_LLMnet.parameters(), lr=init_lr)
def lr_lambda(current_step):
    # 1. Linear Warmup Phase
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    # 2. Cosine Annealing Phase
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    # Clamp progress at 1.0 to avoid errors if training goes longer than expected
    progress = min(1.0, progress)
    cosine_decay = 0.5 * (1.0 + torch.cos(torch.tensor(torch.pi * progress)).item()) # Standard cosine
    # Calculate the ratio between final_lr and init_lr to ensure we hit the floor
    lr_ratio = final_lr / init_lr
    # The multiplier starts at 1.0 and ends at lr_ratio
    return lr_ratio + (1.0 - lr_ratio) * cosine_decay
# Scheduler
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# Train network to predict response from prompt
start = time.time()
loss_mean_batch = -1.0
for epoch in range(num_epochs): # number of epochs
    list_prompt_response_idx = torch.arange(num_prompt_response).to(device) # initialize the list of prompt+response indices
    running_loss = 0.0 # tracking total loss value
    running_batch = 0 # tracking total number of btaches
    for idx_batch in range(num_batch): # number of batches in one epoch
        if not idx_batch%100 and idx_batch>0:
            print(f'idx_batch : {idx_batch}, lr : {optimizer.param_groups[0]["lr"]:.7f}, time(min) : {(time.time()-start)/60:.3f}')
        if not idx_batch%500 and idx_batch>0:
            loss_mean_batch = running_loss / running_batch
            print(f'-> loss_mean_batch : {loss_mean_batch:.4f}')
            running_loss = 0.0
            running_batch = 0
        batch_idx, list_prompt_response_idx = get_batch(batch_size, list_prompt_response_idx) # sample a batch of indices (prompt,response)
        score_seq, target_seq, predicted_seq = SFT_LLMnet(batch_idx.to(device), list_prompt, list_response) # predict next tokens, size=[num_tokens_for_loss, num_tokens], [num_tokens_for_loss], [batch_size, len_response]
        loss = nn.CrossEntropyLoss()(score_seq, target_seq) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        running_batch += 1
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        # Stopping condition
        if idx_batch>500 and loss_mean_batch < 0.1:
            print("\n loss value is small -- training stopped\n")
            break
    if not epoch%1: 
        print('Epoch: %d, time(min): %.3f, lr= %.6f, loss_mean_batch: %.3f' % (epoch, (time.time()-start)/60, optimizer.param_groups[0]['lr'], loss_mean_batch) )
        # print one prompt
        idx_prompt = 0
        print('prompt        :',func_tokens2str(func_indices2tokens(list_prompt[batch_idx[idx_prompt]].tolist())))
        print('response      :',func_tokens2str(func_indices2tokens(list_response[batch_idx[idx_prompt]].tolist())))
        print('predicted_seq :',func_tokens2str(func_indices2tokens(predicted_seq[idx_prompt,:][:list_response[batch_idx[idx_prompt]].size(0)].tolist())),'\n' )
        # save checkpoint
        torch.save({
            'epoch': epoch,
            'tot_time': time.time()-start,
            'loss': loss_mean_batch,
            'net_parameters': net_parameters,
            'SFT_LLMnet_dict': SFT_LLMnet.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            }, '{}.pkl'.format(checkpoint_dir + "/step2_checkpoint_SFT_LLM_" + time_stamp ))


# GPU training time : Epoch: 8, time(min): 9.769, lr= 0.000300, loss_epoch: 0.050


NVIDIA RTX A5000
device: cuda 

checkpoint_file: checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl 

Load pre-trained SSL-LLM: 
 checkpoint file: checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl
 epoch: 0, time: 2278.492min, loss=1.0443
 num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6

num_tokens: 129, padding_int: 127, eos_int: 128

num_net_parameters: 10761345 / 10.76 million

checkpoint file : checkpoint/step2_checkpoint_SFT_LLM_26-02-23--15-37-33.pkl 

num_prompt_response: 200000, batch_size: 100, num_batch: 2000

num_epochs:  1 

idx_batch : 100, lr : 0.0003000, time(min) : 0.823
idx_batch : 200, lr : 0.0002980, time(min) : 1.675
idx_batch : 300, lr : 0.0002919, time(min) : 2.529
idx_batch : 400, lr : 0.0002820, time(min) : 3.382
idx_batch : 500, lr : 0.0002685, time(min) : 4.235
-> loss_mean_batch : 3.9287
idx_batch : 600, lr : 0.0002518, time(min) : 5.088
idx_batch : 700, lr : 0.0002323, time(min) : 5.939
idx_batch : 

## Load pre-trained SFT-LLM network


In [8]:
# GPU 
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec
        
# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training 
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation 
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

# class of supervised fine-tuning LM network (step 2)
class SFT_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int #torch.tensor([eos_int]).to(device)
        self.eos = eos_int #torch.tensor([eos_int]).to(device)
    def forward(self, batch_idx, list_prompt, list_response): # batch_idx.size=[batch_size], len(list_prompt,list_response), =[num_prompt_response]
        prompts = [list_prompt[idx] for idx in batch_idx] # sample list of prompts, len(prompts)=num_prompt_response
        len_prompt = max([len(prompt) for prompt in prompts]) # compute max of prompt lengths
        responses = [list_response[idx] for idx in batch_idx] # sample list of responses, len(prompts)=num_prompt_response
        len_response = max([len(response) for response in responses]) # compute max of response lengths
        batch_size = batch_idx.size(0)
        predicted_seq = torch.ones(batch_size, max(len_prompt,self.context_length)).long().to(device) * self.padding # initiliaze with padding
        for idx in range(batch_size): predicted_seq[idx, -prompts[idx].size(0):] = prompts[idx] # fill batch_predicted_seq with prompt, right-aligned
        predicted_seq_scores = []
        for idx in range(len_response+1): # number of auto-regressive prediction
            context = predicted_seq[:,-self.context_length:] # size=[batch_size, context_length
            H = self.token2vec(context) + self.PE_embedding(self.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
            for transformer_block in self.transformer_blocks: H = transformer_block(H) # size=(batch_size, context_length, d)
            token_scores = H[:,-1,:] # extract last token to predict the next one, size=[batch_size, d]
            token_scores = self.token_prediction(token_scores) # compute scores, size=[batch_size, num_tokens]
            token_probs = torch.softmax(token_scores, dim=1) # compute probs, size=[batch_size, num_tokens]
            next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[batch_size, 1]
            predicted_seq = torch.cat((predicted_seq, next_token), dim=1) # size=[batch_size, current_seq_len+1]
            predicted_seq_scores.append(token_scores.unsqueeze(1)) # append size=[batch_size, 1, num_tokens]
        predicted_seq_scores = torch.cat(predicted_seq_scores, dim=1) # size=[batch_size, len_response+1, num_tokens]
        score_seq = []
        for idx in range(batch_size): score_seq.append(predicted_seq_scores[idx,:responses[idx].size(0)+1,:]) # append size=[len_response_idx, num_tokens]
        score_seq = torch.cat(score_seq, dim=0) # [num_tokens_for_loss=sum_idx len_response_idx+1, num_tokens]
        target_seq = [] #torch.ones(batch_size, len_response+1).long().to(device) * self.eos # init with padding value
        for idx in range(batch_size):  target_seq.append(responses[idx].to(device)); target_seq.append(self.eos); # append size=[len_response_idx]
        target_seq = torch.cat(target_seq, dim=0) # [num_tokens_for_loss]
        predicted_seq = predicted_seq[:,max(len_prompt,self.context_length):] # size=[batch_size, max(len_prompt,context_length)]
        return score_seq, target_seq, predicted_seq # return prediction sequences and scores w.r.t. the dictionary of tokens

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param
    
# load pre-trained SFT-LM network
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')
# checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_' + time_stamp + '.pkl'
import os
checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl' 
print('checkpoint_file:', checkpoint_file, '\n')
import os
os.makedirs('checkpoint', exist_ok=True)
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/mj7gv6bsg1muo8klmuala/step2_checkpoint_SFT_LLM_26-02-07-18-48-32_200K_10M.pkl?rlkey=ht81s1zejbs49k9ffls8biry7&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}"
checkpoint = torch.load(checkpoint_file, map_location=device)
epoch = checkpoint['epoch']
tot_time = checkpoint['tot_time']
loss = checkpoint['loss']
print('Load pre-trained SFT-LM: \n checkpoint file: {:s}\n epoch: {:d}, time: {:.3f}min, loss={:.4f}'.format(checkpoint_file,epoch,tot_time,loss))
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
print(' num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, context_length, num_heads, num_layers) )
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
print('num_tokens: %d, padding_int: %d, eos_int: %d\n' % (num_tokens, padding_int, eos_int))
SFT_LLMnet = SFT_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int)
SFT_LLMnet = SFT_LLMnet.to(device)
SFT_LLMnet.load_state_dict(checkpoint['SFT_LLMnet_dict']) # load pre-trained SFT-LM network from step #2
num_param = number_param(SFT_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )


device: cpu 

device: cpu 

checkpoint_file: checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl 

Load pre-trained SFT-LM: 
 checkpoint file: checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl
 epoch: 0, time: 748.458min, loss=0.0258
 num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6

num_tokens: 129, padding_int: 127, eos_int: 128

num_net_parameters: 10761345 / 10.76 million



##  Generate a response from a training prompt


In [39]:
# generate new sentence of any length
def generate(LLMnet, prompt, max_length_gen_seq, temperature=1.0):
    LLMnet.eval()
    predicted_seq = torch.ones(1, max(prompt.size(0),LLMnet.context_length)).long().to(device) * LLMnet.padding # initiliaze with padding
    predicted_seq[:, -prompt.size(0):] = prompt # fill batch_predicted_seq with prompt, right-aligned
    for k in range(max_length_gen_seq):
        context = predicted_seq[:,-LLMnet.context_length:] # size=[batch_size, context_length
        H = LLMnet.token2vec(context) + LLMnet.PE_embedding(LLMnet.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
        for transformer_block in LLMnet.transformer_blocks: H = transformer_block(H) # size=(batch_size, context_length, d)
        token_scores = H[:,-1,:] # extract last token to predict the next one, size=[batch_size, d]
        token_scores = LLMnet.token_prediction(token_scores) # compute scores, size=[batch_size, num_tokens]
        token_probs = torch.softmax(token_scores/temperature, dim=1) # compute probs, size=[batch_size, num_tokens]
        next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[batch_size, 1]
        #next_token = torch.max(token_probs, dim=1).indices[0].view(1,1) # size=(1,1)
        if next_token==LLMnet.eos:
            break
        predicted_seq = torch.cat((predicted_seq, next_token), dim=1) # size=[batch_size, current_seq_len+1]
    gen_seq = predicted_seq[0][max(prompt.size(0),LLMnet.context_length):]
    return gen_seq

# print one prompt
num_prompt_response = len(list_prompt) # number of prompt+response sequences
idx_prompt = torch.randint(low=0, high=num_prompt_response, size=(1,)).item() # random number in {0,...,num_prompt_response-1}
print('idx_prompt :',idx_prompt)
prompt = list_prompt[idx_prompt]
print('prompt     :',func_tokens2str(func_indices2tokens(prompt.tolist())))
response = list_response[idx_prompt]
print('response   :',func_tokens2str(func_indices2tokens(response.tolist())))
max_length_gen_seq=15; temperature = 3
gen_seq = generate(SFT_LLMnet, prompt, max_length_gen_seq, temperature)
print('gen_seq    :',func_tokens2str(func_indices2tokens(gen_seq.tolist())))


idx_prompt : 29
prompt     : generate an arithmetic series with 12 terms starting with value 62 and common difference 1
response   : 62 63 64 65 66 67 68 69 70 71 72 73
gen_seq    : 62 66 61 62 66 68 65 70 71 71 72 71


##  Generate a response from a new/unseen prompt


In [41]:
# parameters for arithmetic series
m = max_value # maximum value in the sequence
s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
#print('max_value: %d, start_value: %d, common_difference: %d, number_of_terms: %d' % (max_value,s,d,n))

# generate prompt : sample a prompt between 3 candidate prompts
prompt = {}
prompt[1] = 'generate an arithmetic series with ' + str(n) + ' terms starting with value ' + str(s) \
            + ' and common difference ' + str(d)
prompt[2] = 'make a series of arithmetic type which starts at ' + str(s) + ' with ' + str(n) + ' elements and ' + str(d)\
            + ' common difference value'
prompt[3] = 'Let ' + str(n) + ' be the number of terms ' + str(s) + ' the starting number and ' + str(d)\
            + ' the common difference then write the arithmetic series'
random_int = torch.randint(low=1, high=3+1, size=(1,)).item() # random number in {1,2,3}
prompt = prompt[random_int]

response = arithmetic_series(max_value,s,d,n)

# covert from token to integrer
prompt = [str(i) for i in prompt.split()] # convert a string into seq of tokens (w/ string type)
prompt = func_tokens2indices(prompt) # convert from token (str) to index (int)
prompt = torch.tensor(prompt) # convert to pytorch
response = [str(i) for i in response] # convert a string into seq of tokens (w/ string type)
response = func_tokens2indices(response) # convert from token (str) to index (int)
response = torch.tensor(response) # convert to pytorch
max_length_gen_seq=15; temperature = 3
gen_seq = generate(SFT_LLMnet, prompt, max_length_gen_seq, temperature)
print('prompt     :', func_tokens2str(func_indices2tokens(prompt.tolist())))
print('response   :', func_tokens2str(func_indices2tokens(response.tolist())))
print('gen_seq    :',func_tokens2str(func_indices2tokens(gen_seq.tolist())))


prompt     : generate an arithmetic series with 5 terms starting with value 3 and common difference 4
response   : 3 7 11 15 19
gen_seq    : 5 6 11 15 19
